In [1]:
print("hello")

hello


In [2]:
import pandas as pd
from pathlib import Path

In [3]:
DATA_PATH = Path("../data/raw")

customers = pd.read_csv(DATA_PATH / "customers.csv")
orders = pd.read_csv(DATA_PATH / "orders.csv")
order_items = pd.read_csv(DATA_PATH / "order_items.csv")
payments = pd.read_csv(DATA_PATH / "order_payments.csv")
reviews = pd.read_csv(DATA_PATH / "order_reviews.csv")
products = pd.read_csv(DATA_PATH / "products.csv")
sellers = pd.read_csv(DATA_PATH / "sellers.csv")
geolocation = pd.read_csv(DATA_PATH / "geolocation.csv")
category = pd.read_csv(DATA_PATH / "product_category_name_translation.csv")

- convert date columns

In [4]:
date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

In [5]:
for col in date_columns["orders"]:
    orders[col] = pd.to_datetime(orders[col])

for col in date_columns["order_items"]:
    order_items[col] = pd.to_datetime(order_items[col])

for col in date_columns["reviews"]:
    reviews[col] = pd.to_datetime(reviews[col])

In [6]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


- clean the data

In [7]:
# -----------------------------
# DATA CLEANING
# -----------------------------

# 1. Remove duplicate geolocation records
geolocation = geolocation.drop_duplicates()

# 2. Remove products with missing physical dimensions
products = products.dropna(
    subset=[
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
)

print("Data cleaning completed!\n")

print(f"Geolocation rows after removing duplicates : {len(geolocation)}")
print(f"Products rows after removing incomplete records : {len(products)}")

Data cleaning completed!

Geolocation rows after removing duplicates : 738332
Products rows after removing incomplete records : 32949


- feature engineering

In [8]:
# -----------------------------
# FEATURE ENGINEERING
# -----------------------------

# Delivery time (in days)
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] -
    orders["order_purchase_timestamp"]
).dt.days

# Order approval time (in days)
orders["approval_days"] = (
    orders["order_approved_at"] -
    orders["order_purchase_timestamp"]
).dt.days

# Purchase Year
orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year

# Purchase Month Number
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month

# Purchase Month Name
orders["purchase_month_name"] = orders["order_purchase_timestamp"].dt.month_name()

print("✅ Feature Engineering Completed!")

# Preview new columns
orders[[
    "order_purchase_timestamp",
    "delivery_days",
    "approval_days",
    "purchase_year",
    "purchase_month",
    "purchase_month_name"
]].head()

✅ Feature Engineering Completed!


,order_purchase_timestamp,delivery_days,approval_days,purchase_year,purchase_month,purchase_month_name
0,2017-10-02 10:56:33,8.0,0.0,2017,10,October
1,2018-07-24 20:41:37,13.0,1.0,2018,7,July
2,2018-08-08 08:38:49,9.0,0.0,2018,8,August
3,2017-11-18 19:28:06,13.0,0.0,2017,11,November
4,2018-02-13 21:18:39,2.0,0.0,2018,2,February


- Save processed data

In [ ]:
from pathlib import Path

# Create processed folder if it doesn't exist
processed_path = Path("../data/processed")
processed_path.mkdir(exist_ok=True)

# Store all cleaned datasets
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category": category
}

# Save each dataset
for name, df in datasets.items():
    df.to_csv(processed_path / f"{name}.csv", index=False)

print("All processed datasets saved successfully!")

print("\nSaved Files:")
for file in processed_path.iterdir():
    print(file.name)

✅ All processed datasets saved successfully!

Saved Files:
category.csv
customers.csv
geolocation.csv
orders.csv
order_items.csv
payments.csv
products.csv
reviews.csv
sellers.csv
